# REFCON quickstart

Reference-free per-gene **copy-number** inference from scRNA-seq. This notebook: installs the
package, fetches the released checkpoints + paired single-cell-DNA cohorts, runs REFCON on one
cohort (no reference/normal cells needed), scores it against paired scWGS, and visualizes the
per-cell copy-number calls. Runs on GPU, Apple MPS, or CPU.

## 1. Install
In Colab or a fresh clone, uncomment the clone line. `pip install -e .` reads `pyproject.toml`.

In [ ]:
# !git clone https://github.com/gencturkmert/REFCON.git
# %cd REFCON
%pip install -e . --quiet
%pip install matplotlib --quiet   # for the plots in step 5

## 2. Fetch checkpoints + data
Downloads the three ensemble checkpoints and the A375 / HCT116 cohorts from Zenodo
(no-op if already present).

In [ ]:
!python scripts/fetch_data.py

## 3. Run REFCON inference (reference-free)
Load one released checkpoint and infer per-gene CN ratios directly from expression.
The paper averages three checkpoints (`ens3`); one checkpoint is enough for a demo.

In [ ]:
import numpy as np, torch, anndata as ad, scipy.sparse as sp
from refcon.model_binned_dev import BinnedDevModel
from refcon.eval_cellwide import get_chr_ranges, run_cellwide_inference, broadcast_bins_to_genes
from refcon.metrics import per_cell_metrics

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

ckpt = torch.load('checkpoints/baseline_bs8_mc1024.pt', map_location=device, weights_only=False)
cfg = ckpt['config']
model = BinnedDevModel(bin_size=cfg['bin_size'], max_chunk=cfg['max_chunk'],
                       d_model=cfg.get('d_model', 128), n_heads=cfg.get('n_heads', 4),
                       n_layers=cfg.get('n_layers', 4), dim_ff=cfg.get('dim_ff', 512), dropout=0.0).to(device)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
print('checkpoint params:', f"{sum(p.numel() for p in model.parameters()):,}")

# HCT116 colorectal cohort (raw UMI) + paired scWGS ground truth
adata = ad.read_h5ad('data/hct116/hct116_dntr_per_cell.h5ad')
gt = np.load('data/hct116/hct116_dntr_per_cell_cn.npy')      # per-cell absolute CN, row-aligned
adata = adata[:96].copy(); gt = gt[:96]      # 96-cell demo (~1 min on a laptop); drop for all 1,468

X = adata.X; X = X.toarray() if sp.issparse(X) else np.asarray(X)
X_log = np.log1p(X.astype('float32'))        # REFCON input = log1p(raw UMI); no normalization
res = run_cellwide_inference(model, X_log, get_chr_ranges(adata), torch.device(device), cell_batch=8)
pred = broadcast_bins_to_genes(res['bin_profile'], res['bin_coords'], adata.n_vars)
print('predicted CN-ratio matrix:', pred.shape, '(per-cell mean approx. 1)')

## 4. Score against paired single-cell DNA
The paper's exact per-cell metric (`refcon.metrics.per_cell_metrics`).

In [ ]:
m = per_cell_metrics(pred, gt.astype(float), fill_diploid=True)
print(f'per-cell Pearson = {np.nanmean(m.P):.3f}')
print(f'per-cell AUROC   = loss {np.nanmean(m.AL):.3f} | gain {np.nanmean(m.AG):.3f}')

## 5. Visualize cell-level copy-number calls

In [ ]:
import matplotlib.pyplot as plt
order = np.argsort(adata.var['abspos'].values)     # genomic order
c = int(np.nanargmax(m.P))                          # a well-correlated cell

fig, ax = plt.subplots(2, 1, figsize=(11, 6))
ax[0].plot(pred[c][order], lw=.6, label='REFCON (predicted CN ratio)')
ax[0].plot(gt[c][order] / np.nanmedian(gt[c]), lw=.8, alpha=.6, label='scWGS ground truth')
ax[0].axhline(1, color='k', lw=.4, ls=':')
ax[0].set(title=f'HCT116 cell {c} - genome-wide CN (Pearson {m.P[c]:.2f})',
          xlabel='gene (genomic order)', ylabel='CN ratio'); ax[0].legend(loc='upper right')
im = ax[1].imshow(pred[:, order], aspect='auto', cmap='bwr', vmin=0.5, vmax=1.5, interpolation='nearest')
ax[1].set(title='per-cell copy-number calls (red = gain, blue = loss)',
          xlabel='gene (genomic order)', ylabel='cell')
fig.colorbar(im, ax=ax[1], fraction=.02, label='CN ratio')
plt.tight_layout(); plt.show()

## Next steps
- **Full cohort + released ensemble (paper numbers):**
  `python scripts/infer.py --data data/hct116/hct116_dntr_per_cell.h5ad --out out/hct116`
  averages all three checkpoints over all 1,468 cells, then score with
  `python scripts/score.py --pred out/hct116/ens3_cn.npz --gt data/hct116/hct116_dntr_per_cell_cn.npy`.
- **A375:** swap the paths to `data/a375/a375_dntr_per_cell*`.
- **Your own data:** any `.h5ad` with raw UMI in `X` and `var` carrying `chromosome` / `start` / `end` /
  `chr_boundary` (see `docs/input_format.md`). Reference-free; no normal cells required.